*Notebook Last Validated: 2026-09-11*

Project UID (Internal, Prod): 0c41bbf9-4acf-4a62-9235-090535437cab

# Encrypted Model Code and Weights on Rhino FCP

Trains and runs inference with the `hello-pt-secure` NVFlare example on Rhino FCP, encrypting both the model code (`network.py`) and the trained model weights with a key only you hold.

See `README.md` in this directory for a full explanation of how the encryption works, and for the local (non-FCP, Docker-only) walkthrough.

#### Prerequisites
1. `pip install rhino_health cryptography`
2. Access to a Rhino FCP workgroup and its container registry - see the "Find your workgroup's container registry" section below

## Setup

In [1]:
import json
import os
import subprocess
from getpass import getpass

import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeTypes,
    ModelTrainInput,
)
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput
from rhino_health.lib.endpoints.project.project_dataclass import ProjectCreateInput

# Pin the working directory to this notebook's folder so relative paths resolve
# regardless of where the notebook is opened from (e.g. VS Code, Jupyter Lab, etc.)
os.chdir(os.path.dirname(os.path.abspath(globals().get('__vsc_ipynb_file__', 'notebook.ipynb'))))
print("Setup Complete")

Setup Complete


## Generate and encrypt the model code

Unlike a standard NVFlare example, this one needs its own encryption key before anything else will run, since the `network.py.enc` shipped in `custom/` was encrypted with a key you don't have. This cell generates a new key (reusing it if you already have one at `~/myprecious`) and re-encrypts `network.py` with it.

In [ ]:
KEY_PATH = os.path.expanduser("~/myprecious")

if not os.path.exists(KEY_PATH):
    print("Generating encryption key...")
    subprocess.run(["python", "encrypt_code/generate_key.py", KEY_PATH], check=True)
else:
    print(f"Using existing encryption key at {KEY_PATH}")

print("Encrypting network.py...")
subprocess.run(
    ["python", "encrypt_code/encrypt_code.py", "network.py", KEY_PATH, "app/custom/network.py.enc"],
    check=True,
)

with open(KEY_PATH) as f:
    encryption_key = f.read()
print("Done")

print(
    "IMPORTANT: build/push your container image AFTER this cell, from this exact state of "
    "app/custom/network.py.enc. If you already built an image before running this cell (or with a "
    "different ~/myprecious), rebuild and re-push it now, or training will fail with an opaque "
    "'NVFLARE server and clients failed to connect' error (the container crashes on decrypt "
    "before the NVFlare client ever starts)."
)

## Find your workgroup's container registry, then build and push the image

> **Do this every time you (re)build the image**, using whatever `app/custom/network.py.enc` / `~/myprecious` the cell above just produced. Reusing an older, already-pushed image - or building before running the cell above - bakes in a mismatched key: the container will crash on decrypt before NVFlare ever starts, which surfaces only as a generic "server and clients failed to connect" timeout with no other error.

FCP runs your own pre-built image rather than building one from source, so you build and push it *outside* this notebook, using the shared push script in `user-resources/utils/`:

1. **Find your workgroup's container repository name** - in the FCP UI, go to **Settings (gear icon) -> Containers & Artifacts -> Workgroup container registry** (there's a copy button next to it).
2. See [Pushing Containers to the ECR](https://docs.rhinofcp.com/getting-started/quick-start-guide/pushing-containers-to-the-ecr) and make sure you've completed all the pre-requisites
3. **Build and push**, from this directory, by entering the following in your command line (replace the elements in <>):
   ```bash
   ../../../utils/docker-push.sh <your-workgroup-repo-name> <a-tag-you-choose>
   ```
   Example: `../../../utils/docker-push.sh workgroup-rhino-health-prod DO-333-test-v5`
   
   This can take some time and will print `Done. Container image URI: ...` when it finishes - **copy that exact URI** into `CONTAINER_IMAGE_URI` below (it's what you actually pushed, so it's more reliable than reconstructing the URI yourself).

In [ ]:
CONTAINER_IMAGE_URI = "865551847959.dkr.ecr.us-east-1.amazonaws.com/workgroup-rhino-health-prod:DO-333-test-v5" # UPDATE THIS TO THE URI PRINTED BY docker-push.sh

if CONTAINER_IMAGE_URI.startswith("<"):
    raise ValueError("Set CONTAINER_IMAGE_URI to the URI printed by docker-push.sh before continuing.")

print("Completed.")

Completed.


## Authentication

Log in to the Rhino FCP. When prompted, provide your password.

**Variables to adjust:**
- `USERNAME` - your Rhino FCP username (typically your email)

In [ ]:
USERNAME = "<YOUR_RHINO_FCP_USERNAME>"  # REPLACE WITH YOUR RHINO FCP USERNAME

print("Logging In")
session = rh.login(username=USERNAME, password=getpass())
user = session.current_user
print("Logged In")

Logging In
Logged In


## Project Creation

Creates a new project on the Rhino FCP under the current user's primary workgroup.

Note that every time you rerun this cell, it creates a NEW project (a different `project_uid`), even with the same `PROJECT_NAME`.

In [ ]:
PROJECT_NAME = "[User-Example] Encrypted Model Code and Weights Demo"

project = session.project.add_project(
    ProjectCreateInput(
        name=PROJECT_NAME,
        description="Encrypted NVFlare model training and inference",
        type="Validation",
        primary_workgroup_uid=user.primary_workgroup_uid,
    )
)
print(f"Created project: {project.name}")

Created project: Encrypted Model Code and Weights Demo


## Dataset Creation

Registers the placeholder training images (`data/train/NORMAL`, `data/train/PNEUMONIA`) as a Dataset on FCP. `CLIENT_DATA_PATH` must be a path your on-prem client can actually see - swap in your own pneumonia CXR data here to use anything beyond the tiny placeholder set.

**Variables to adjust:**
- `CLIENT_DATA_PATH` - base path to this repo (or your own data) on your on-prem client's filesystem

In [6]:
CLIENT_DATA_PATH = "/rhino_data/external/import-external-datasets-dev"  # REPLACE WITH YOUR RHINO FCP CLIENT DATA PATH
FILE_BASE_PATH = "user-resource-examples/pneumonia_cxr/train"  # relative to CLIENT_DATA_PATH
DATASET_NAME = "Pneumonia CXR (training)"

dataset = session.dataset.add_dataset(
    DatasetCreateInput(
        name=DATASET_NAME,
        description="Placeholder pneumonia CXR training data",
        project_uid=project.uid,
        workgroup_uid=project.primary_workgroup_uid,
        file_base_path=os.path.join(CLIENT_DATA_PATH, FILE_BASE_PATH),
        method="filesystem",
        data_schema=None,
        is_data_deidentified=True,
    )
)
print(f"\nFinished Registering Dataset.")
print(f"\nVerify the dataset was registered successfully by visiting the Datasets page in the Rhino FCP UI, and searching for `{DATASET_NAME}`.")
print(f"If you used the dataset we provided, you should see 10 rows in the dataset once registration is finalized, corresponding to the number of files that were uploaded.")


Finished Registering Dataset.

Verify the dataset was registered successfully by visiting the Datasets page in the Rhino FCP UI, and searching for `Pneumonia CXR (training)`.
If you used the dataset we provided, you should see 10 rows in the dataset once registration is finalized, corresponding to the number of files that were uploaded.


## NVFlare Code Object Creation

Creates a Code Object that points at the image you pushed above, rather than having FCP build one from source the way the auto-container examples do.

In [8]:
CODE_OBJ_NAME = "Encrypted Model Code and Weights"
code_object_input = CodeObjectCreateInput(
    name=CODE_OBJ_NAME,
    description="Encrypted NVFlare model training and inference",
    input_data_schema_uids=[None],
    output_data_schema_uids=[None],
    project_uid=project.uid,
    code_type=CodeTypes.NVIDIA_FLARE_V2_6,  # must match requirements.txt: nvflare==2.6.0 - FCP's provisioning only supports up to v2.6
    config={"container_image_uri": CONTAINER_IMAGE_URI},
)
# return_existing=False + add_version_if_exists=True: if a Code Object named CODE_OBJ_NAME
# already exists (e.g. from a prior run), create a NEW VERSION with this config instead of
# silently returning the old one - otherwise a fixed CONTAINER_IMAGE_URI update above would
# never actually take effect on re-runs.
code_object = session.code_object.create_code_object(
    code_object_input, return_existing=False, add_version_if_exists=True
)
code_object = code_object.wait_for_build()  # returns immediately - the image is already built

print(f"Finished Creating Code Object")
print(f"Verify the code object was created successfully by visiting the Code Objects page in the Rhino FCP UI, and searching for `{CODE_OBJ_NAME}`.")

Finished Creating Code Object
Verify the code object was created successfully by visiting the Code Objects page in the Rhino FCP UI, and searching for `Encrypted Model Code and Weights`.


In [9]:
# You can run these cells to inspect the code object and its config, and confirm that the container image URI is correct.
co = session.code_object.get_code_object(code_object.uid)
print(co.config)

{'image_tag': 'DO-333-test-v5', 'image_repo_name_part': 'rhino-health-prod', 'container_image_uri': '865551847959.dkr.ecr.us-east-1.amazonaws.com/workgroup-rhino-health-prod:DO-333-test-v5'}


## NVFlare Federated Training Run

Loads the client/server config files, then kicks off training, passing the encryption key generated above as the federated secret. Waits for the run to complete before proceeding. This can take 15+ min to run the first time.

In [10]:
with open("app/config/config_fed_client.json") as f:
    config_fed_client = json.dumps(json.load(f))
with open("app/config/config_fed_server.json") as f:
    config_fed_server = json.dumps(json.load(f))

run_params = ModelTrainInput(
    code_object_uid=code_object.uid,
    input_dataset_uids=[dataset.uid],
    one_fl_client_per_dataset=True,
    validation_dataset_uids=[],
    validation_datasets_inference_suffix="",
    timeout_seconds=1200,
    config_fed_client=config_fed_client,
    config_fed_server=config_fed_server,
    secrets_fed_client=json.dumps({"key": encryption_key}),
    secrets_fed_server=json.dumps({"key": encryption_key}),
)

model_train = session.code_object.train_model(run_params)
code_run = model_train.wait_for_completion(1200, poll_frequency=30)
print(f"Training finished with status: {code_run.status}")

Waiting for code run to complete (0 hours 0 minutes and a second)
Waiting for code run to complete (0 hours 0 minutes and 32 seconds)
Waiting for code run to complete (0 hours a minute and 4 seconds)
Waiting for code run to complete (0 hours a minute and 35 seconds)
Waiting for code run to complete (0 hours 2 minutes and 6 seconds)
Done.
Training finished with status: CodeRunStatus.COMPLETED


In [11]:
# If the training failed, you can inspect the errors to see what went wrong.
print(code_run.errors)

[]


## Model Parameters Download

Downloads the trained, still-encrypted weights.

In [12]:
weights = session.code_run.get_model_params(code_run.uid)
with open("model_parameters.pt.enc", "wb") as f:
    f.write(weights.getbuffer())
print("Saved encrypted weights to model_parameters.pt.enc")

Saved encrypted weights to model_parameters.pt.enc


## Next Steps: Decrypt and Run Inference Locally

**What's going on:** the weights in `model_parameters.pt.enc`, which you just downloaded, are encrypted - nobody but you (whoever holds the key at `~/myprecious`) can read them. To actually use the trained model - for example, to test it against some images - you first need to decrypt it using that same key.

This example decrypts and tests the model using the `infer.py` script. That script depends on several Python packages (PyTorch, pandas, etc.) that are only installed inside the `hello-pt-secure` Docker image - not on your own computer. So rather than running `infer.py` directly on your machine, you run it *inside* that container, the same way you did for local training earlier.

**What "mounting" a folder means:** a Docker container can't see your computer's files by default. To let it read/write specific folders, you "mount" them when you start the container - you tell it "folder X on my computer should appear as `/input` inside you" (and similarly for `/output`). Anything you put in folder X on your machine becomes visible at `/input` inside the container, and anything the container writes to `/output` shows up back in the folder on your machine that you mounted there.

**Step-by-step:**

1. **Put the encrypted weights where the container will look for them** (its "output" folder, since that's also where training originally wrote them):
   ```bash
   mkdir -p ~/pneumonia-test/output
   cp model_parameters.pt.enc ~/pneumonia-test/output/
   ```

2. **Put some sample images to test against, plus the decryption key, where the container will look for them** (its "input" folder):
   ```bash
   mkdir -p ~/pneumonia-test/infer_mount/file_data
   cp -r data/test/NORMAL data/test/PNEUMONIA ~/pneumonia-test/infer_mount/file_data/
   cp data/dataset_test.csv ~/pneumonia-test/infer_mount/dataset.csv
   KEY=$(cat ~/myprecious)
   echo "{\"key\": \"$KEY\"}" > ~/pneumonia-test/infer_mount/secret_run_params.json
   ```
   This must be the *same* key used to train the model - if training and inference ever use different keys, decryption will fail with an error like `InvalidToken`.

3. **Make sure your local Docker image is up to date.** If you've changed anything in this repo, or re-run the "Generate and encrypt" cell, since you last built it, rebuild now - an out-of-date image can also cause the same `InvalidToken` decryption error, or fail to find files that moved:
   ```bash
   docker build -t hello-pt-secure .
   ```

4. **Start the container**, connecting the two folders from steps 1-2 to its `/input` and `/output`:
   ```bash
   docker run -it \
     -v ~/pneumonia-test/infer_mount:/input \
     -v ~/pneumonia-test/output:/output \
     hello-pt-secure bash
   ```
   This drops you into a command-line shell *inside* the container - your terminal prompt will change to something like `localuser@<container-id>:~$`.

5. **Still inside that shell**, decrypt the weights and score the test images:
   ```bash
   python infer.py /output/model_parameters.pt.enc
   cat /output/dataset.csv   # a Model_Score column should now be added
   ```
Enter `exit` to leave the continer.

You can also skip steps 1-2 above and download the weights directly from the Rhino FCP UI's Code Runs page (click the three dots next to the code run and select "Download Model Parameters") instead of using the download cell earlier in this notebook.


## Cleanup

**On FCP:** this notebook creates a new Project (containing a Dataset and Code Object) every time you run it. Set `CLEANUP = True` below and re-run that cell to remove them once you're done. (This does not delete the pushed container image from your registry - see below.)

**Locally:**
- `~/myprecious` is your encryption key - there is no way to recover the encrypted weights without it, so keep a copy somewhere safe before deleting it, if you want to keep them.
- This notebook overwrites the tracked placeholder `app/custom/network.py.enc` with your own encrypted output, and creates a new, untracked `model_parameters.pt.enc` in this directory. Check `git status` before committing anything in this directory - you likely don't want to commit either of these; `git checkout -- app/custom/network.py.enc` restores the shipped placeholder.
- Remove the local Docker image, if you built one: `docker rmi hello-pt-secure`
- If you also ran the local Docker walkthrough in `README.md`, remove its scratch directory: `rm -rf ~/pneumonia-test`
- (Optional) Delete the pushed image from your registry once you no longer need it, e.g. for ECR: `aws ecr batch-delete-image --repository-name <your-workgroup-repo-name> --image-ids imageTag=<tag>`

In [ ]:
CLEANUP = False  # Set to True to delete the Project (and its Dataset/Code Object) created above

if CLEANUP:
    session.code_object.remove_code_object(code_object)
    session.dataset.remove_dataset(dataset)
    session.project.remove_project(project)
    print("Removed code object, dataset, and project from FCP.")
else:
    print("Skipped - set CLEANUP = True above to remove the Project/Dataset/Code Object this notebook created.")

## Additional Resources

- [Rhino SDK Documentation](https://rhinohealth.github.io/rhino_sdk_docs/html/autoapi/index.html)
- [Rhino User Resources](https://github.com/RhinoHealth/user-resources/tree/main)
- [Rhino FCP Platform Documentation](https://docs.rhinofcp.com/)
- [NVFlare GitHub](https://github.com/NVIDIA/NVFlare/tree/main)